# NeuroGolf Solver Family: Mask / Object Selection

This notebook is a starter pipeline for the `mask_object_selection` task family.

Workflow:

1. Load task ids from `task_groups/task_type_groups.json`.
2. Inspect the family metadata from `task_type_map.csv`.
3. Train or infer a per-task ONNX model with `train_family_task`.
4. Save `taskNNN.onnx` files.
5. Build `submission.zip` from the generated models.

The generated maps are heuristic solver-routing labels. Validate against visible examples before submitting.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if str(ROOT / 'submission_nbs') not in sys.path:
    sys.path.append(str(ROOT / 'submission_nbs'))

from neurogolf_nb_common import *

FAMILY = 'mask_object_selection'
MODEL_VERSION = 'mask-object-selection-v0.1'
DATA_DIR, BASE_OUT_DIR = default_paths()
OUT_DIR = BASE_OUT_DIR / FAMILY
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)

In [ ]:
task_map = load_task_type_map()
task_ids = family_task_ids(FAMILY)
family_df = task_map[task_map.primary_family == FAMILY].copy()

print('family:', FAMILY)
print('tasks:', len(task_ids))
display(family_df.head(20))

In [ ]:
# Inspect one task quickly.
if task_ids:
    sample_task_id = task_ids[0]
    sample_task = load_task(DATA_DIR, sample_task_id)
    print(sample_task_id, 'examples:', len(all_examples(sample_task)))
    print('first input shape:', grid_shape(sample_task['train'][0]['input']))
    print('first output shape:', grid_shape(sample_task['train'][0]['output']))
    print('first input:', sample_task['train'][0]['input'])
    print('first output:', sample_task['train'][0]['output'])
else:
    print('No tasks currently mapped to this family.')

In [ ]:
def train_family_task(task):
    # Starter trainer for same-shape object/mask tasks.
    # First useful sub-solvers: color masks, remove-background, keep-one-color.
    return None, {'ok': False, 'reason': 'mask/object selection ONNX trainer pending'}

In [ ]:
# Dry-run training on the first few tasks without saving.
dry_rows = []
for task_id in task_ids[:10]:
    task = load_task(DATA_DIR, task_id)
    model, info = train_family_task(task)
    dry_rows.append({'task_id': task_id, 'has_model': model is not None, **info})

pd.DataFrame(dry_rows)

In [ ]:
# Build family models.
# Set fallback_identity=True only when you explicitly want placeholder models
# for tasks whose trainer is not implemented yet.
rows, zip_path = build_family_submission(
    FAMILY,
    train_family_task,
    DATA_DIR,
    OUT_DIR,
    fallback_identity=False,
    validate=False,
)

result_df = pd.DataFrame(rows)
display(result_df.head(30))
print('models saved:', int(result_df.get('saved', pd.Series(dtype=bool)).sum()) if len(result_df) else 0)
print('zip:', zip_path)

In [ ]:
# Optional: validate saved ONNX models on visible examples.
# This can be slow for large families and requires onnxruntime.
validate_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({
        'task_id': row['task_id'],
        'right': summary['right'],
        'wrong': summary['wrong'],
    })

pd.DataFrame(validate_rows)

In [ ]:
# Submission helper.
# For a full competition submission, combine models from multiple family
# folders into one directory, then call create_submission_zip(combined_dir).
submission_zip = create_submission_zip(OUT_DIR)
print(submission_zip)